# 06d — `fold_steel_combined`: pooled Steel1+Steel2, built and trained end to end

**What this notebook does.** Everything for one experiment, in one place: it
builds the pooled `fold_steel_combined` split, verifies it, merges it into
`configs/fold_stats.yaml`, pushes it, and then trains on it — the tiling half
of step 3 and the training half of step 6, for this fold only.

The fold itself replaces the leave-one-dataset-out design *for this
experiment*. It pools every Steel1 and Steel2 tile into one set and gives that
pool a fresh 70/15/15 train/val/test split made **by parent**, not by tile.
MetalDam, uhcs1 and uhcs2 take no part in it and are not even indexed here;
their folds, manifests, boundary maps and tiles are untouched on disk.

**What must already exist.**

- `GH_TOKEN` as a host secret and the datasets mounted — `00_bootstrap.ipynb` passes
- `reports/audit.json` — from `01_audit.ipynb`
- `reports/gt_extraction.json` and the boundary PNGs under `GT_BOUNDARIES_ROOT`
  for **Steel1 and Steel2** — from `02_boundary_gt.ipynb`, at `line_width_px=4`.
  This notebook does not regenerate ground truth and does not touch it.
- `configs/fold_stats.yaml` must already exist with the LODO folds in it — this
  notebook MERGES one fold into that file and refuses to create it from scratch,
  because a fresh write from a two-dataset index would delete the other folds.
- `configs/dataloader.yaml` — this host's `num_workers`, from `04_dataset.ipynb`

**What it produces.**

- `reports/manifests/fold_steel_combined.csv` (train+val) and
  `reports/manifests/fold_steel_combined_test.csv` (its own held-out slice)
- one new key, `folds.fold_steel_combined`, in `configs/fold_stats.yaml`
- `reports/tiling_fold_steel_combined.{md,json}` — this fold's own report,
  written beside `reports/tiling.md` rather than over it
- `PERSISTENT_DIR/checkpoints/fold_steel_combined/{last,best}.pt` and
  `reports/train_fold_steel_combined_<platform>.{json,md}`

**Expected runtime on a free T4.** Cells 1–11 (the fold): about 1 minute —
only two datasets are indexed, ~1,400 tiles. Training: the same as any other
fold, hours rather than minutes; on Kaggle it stops cleanly at the session
budget and resumes next session.

## Cell 1 — the standard bootstrap block

Identical in every notebook in this repo, and deliberately so: it reads
`GH_TOKEN` from whichever secret store the host provides, fetches
`scripts/bootstrap_session.py` from the GitHub API, and hands over to
`bootstrap()`, which clones the repo, installs what is missing, resolves every
path for this platform and returns `PATHS`.

Nothing below this cell tests which platform it is running on. Colab and Kaggle
differ in where the data is, whether the session outlives the run, and which
secret store exists — all of that is settled here and in `configs/<platform>.yaml`.

`BRANCH = "main"` on purpose, including on Kaggle: the bootstrap clones `main`,
reads `session.kaggle.branch`, and switches the checkout itself.

In [ ]:
# --- standard bootstrap block: identical in every notebook ---------------
OWNER, REPO, BRANCH = "arhorri", "boundary", "main"

import importlib, os, pathlib, sys, urllib.request


def _gh_token():
    """Read GH_TOKEN from whichever secret store this host provides."""
    try:
        from google.colab import userdata

        return userdata.get("GH_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    return os.environ.get("GH_TOKEN")


_token = _gh_token()
if not _token:
    raise SystemExit(
        "GH_TOKEN secret is missing.\n"
        "  Colab : key icon in the left sidebar -> add GH_TOKEN -> notebook access ON\n"
        "  Kaggle: Add-ons -> Secrets -> add GH_TOKEN -> attach to this notebook"
    )

_req = urllib.request.Request(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/scripts/bootstrap_session.py?ref={BRANCH}",
    headers={
        "Authorization": f"Bearer {_token}",
        "Accept": "application/vnd.github.raw",
    },
)
pathlib.Path("bootstrap_session.py").write_bytes(urllib.request.urlopen(_req).read())
del _token

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
import bootstrap_session

bootstrap_session = importlib.reload(bootstrap_session)

PATHS = bootstrap_session.bootstrap(
    repo_url=f"https://github.com/{OWNER}/{REPO}.git", branch=BRANCH
)

## Cell 2 — what this fold is, and why indexing only two datasets is safe here

`fold_steel_combined` asks a different question from the LODO folds. LODO
measures transfer to an unseen *imaging setup*: a whole dataset the model never
trained on. This fold pools Steel1 and Steel2 and asks whether the model
generalises to unseen *parents* of the same two microscopes.

Three properties of it are load-bearing, and each is checked later rather than
assumed:

1. **The split is by parent.** Steel1's 19 micrographs and Steel2's 4 each
   produce many near-duplicate 256 px tiles. Letting two tiles of one
   micrograph land on opposite sides of the split would leak, and the
   validation score would be measuring memorisation. This is the same
   protection the existing Steel1 parent-split already exists to provide,
   extended across the pool.
2. **The 70/15/15 target is measured in TILES, not parents.** Steel1 averages
   ~47 tiles per parent and Steel2 ~126, so a parent-count split and a
   tile-count split disagree, and tiles are what an epoch actually sees.
3. **Steel2 is guaranteed a parent in every split.** It has only 4 parents in
   total, so leaving its placement to chance could put it entirely on one
   side. `min_parents_per_split` hands each split one of its parents before
   anything else is decided.

Steel2 is globally `test_only` (`tiling.test_only`), meaning it never appears in
any LODO fold's train or val split. This fold pools it with Steel1 anyway. That
is a deliberate, self-contained exception — sound only because this fold carves
out its own genuinely disjoint, parent-level test slice rather than reusing
Steel2's "held out" status as licence to peek.

### Why only Steel1 and Steel2 are indexed, and what that forces

Indexing only the two datasets this fold uses is faster and, more importantly,
means nothing belonging to MetalDam, uhcs1 or uhcs2 is read or rewritten. But a
partial index changes what may safely be written:

| output | full run (`03_tiling.ipynb`) | this notebook |
| --- | --- | --- |
| `reports/manifests/fold_*.csv` | all folds rewritten | only this fold's two files written |
| `reports/manifests/test.csv` | rewritten | **not touched** — it is the LODO folds' Steel2-whole test set and answers a different question |
| `configs/fold_stats.yaml` | rewritten wholesale | **merged** — one key added, every other fold left byte-identical |
| `reports/tiling.{md,json}`, `parents.md` | rewritten | **not touched** — they describe all five datasets; regenerating them from two would silently shrink them |

That third row is the dangerous one. `tiling.write_fold_stats` builds the whole
document from one run's stats and calls `write_text`; running it here would
delete `fold_MetalDam`, `fold_uhcs1`, `fold_uhcs2`, `dev` and `test` from the
file training reads, and nothing would raise until something later asked for a
fold that had quietly vanished. `tiling.merge_fold_stats` is the read-modify-write
that makes a partial run safe, and this notebook uses it.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

from src import boundary_gt
from src import tiling

REPO = Path(PATHS["repo_root"])
REPORTS = Path(PATHS["reports_dir"])
CONFIGS = REPO / "configs"
GT_ROOT = Path(PATHS["gt_boundaries_root"])

settings = tiling.load_config()
steel_cfg = settings["steel_combined"]
FOLD = steel_cfg["name"]
DATASETS = list(steel_cfg["datasets"])
MANIFEST_DIR = REPORTS / settings["manifest_subdir"]

audit = boundary_gt.load_audit(REPORTS)
extraction = tiling.load_extraction(REPORTS)

# The width that ACTUALLY produced the boundary maps, read back rather than
# taken from whatever configs/default.yaml currently says -- pos_weight is
# compared against a band scaled by it further down.
line_width = float(extraction["settings"]["line_width_px"])

print(f"platform        {PATHS['platform']}")
print(f"repo            {REPO}")
print(f"GT boundaries   {GT_ROOT}")
print(f"manifests       {MANIFEST_DIR}")
print()
print(f"fold            {FOLD}")
print(f"datasets        {DATASETS}")
print(f"target ratio    {steel_cfg['train_frac']}/{steel_cfg['val_frac']}"
      f"/{steel_cfg['test_frac']} (train/val/test, measured in TILES)")
print(f"min parents     {steel_cfg['min_parents_per_split']} per dataset per split")
print(f"seed            {settings['seed']}   (configs/default.yaml tiling.seed)")
print(f"held_out label  {steel_cfg['held_out_label']!r}")
print(f"patch/stride    {settings['patch_size']} / {settings['stride']}")
print(f"line_width_px   {line_width}  (from reports/gt_extraction.json, not the config)")

missing = [d for d in DATASETS if d not in (extraction.get("datasets") or {})]
if missing:
    raise SystemExit(
        f"{missing} have no boundary maps in reports/gt_extraction.json. "
        "Run notebooks/02_boundary_gt.ipynb for them on Colab first -- this "
        "notebook never generates ground truth.")
print(f"\nboundary maps present for {DATASETS}: OK")

## Cell 3 — index the tiles of Steel1 and Steel2

`tiling.build_index` opens every boundary PNG once to measure each tile's
boundary fraction. Nothing is resized and no tile image is written: a tile is a
row, and the loader crops it from the full image at load time.

`datasets=DATASETS` is what keeps this to the two datasets this fold uses. Both
are already exactly 256×256 and are declared in `tiling.assert_single_tile`, so
each source image must yield exactly one tile with zero padding — asserted
inside `index_dataset`, which raises if it ever stops being true rather than
quietly padding.

In [ ]:
from tqdm.auto import tqdm


def progress(seq, desc=""):
    return tqdm(seq, desc=desc, leave=False, unit="img")


index = tiling.build_index(extraction, audit, GT_ROOT, settings,
                           datasets=DATASETS, progress=progress)

rows = []
for name, d in index["datasets"].items():
    counts = [p["tiles"] for p in tiling.parent_map(d["tiles"])[name].values()]
    rows.append({
        "dataset": name,
        "parents": d["n_parents"],
        "images": d["n_images"],
        "tiles": d["n_tiles"],
        "tiles/parent": f"{min(counts)}/{int(np.median(counts))}/{max(counts)}",
        "dropped": d["n_dropped"],
        "excluded": d["n_excluded"],
        "no mask": d["n_orphans"],
        "padded px": d["padded_pixels"],
    })
print(pd.DataFrame(rows).to_string(index=False))

fabricated = sum(d["padded_pixels"] for d in index["datasets"].values())
print(f"\nfabricated pixels: {fabricated} "
      f"({'every tile is real image data' if not fabricated else 'SOME IMAGE WAS PADDED -- investigate'})")
for d in index["datasets"].values():
    for loud in d["single_tile_drops"]:
        print(f"  ! {loud['dataset']}/{loud['image']} lost its only tile "
              f"({loud['boundary_fraction']:.5f} boundary) -- it is in no split")

## Cell 4 — build the pooled fold

`tiling.build_steel_combined_fold` does the whole allocation. It is a
self-contained function: it never calls `build_folds()` and `build_folds()`
never calls it, which is why running it against a two-dataset index works at
all — `build_folds()` would raise here, because none of the LODO datasets are
present.

The allocation is deterministic in two stages, both driven by one
`random.Random(tiling.seed)` and both iterating in sorted order so the result
never depends on dict or set iteration order:

1. **Minimum guarantee.** Each dataset hands `min_parents_per_split` parents to
   each of train/val/test before anything else is decided. This is what puts a
   Steel2 parent in every split.
2. **The rest.** Every remaining parent, pooled across both datasets, is
   shuffled once and then assigned one at a time to whichever split is
   currently furthest *below* its target tile share. Greedy, and never moves a
   parent once placed — which is what makes stage 2 reproducible.

The per-dataset, per-split parent and tile counts are printed below rather than
summarised into a total, so an imbalance is visible instead of absorbed.

In [ ]:
fold = tiling.build_steel_combined_fold(index, settings)

alloc = fold["parent_allocation"]
rows_by_split = {
    "train": [r for r in fold["rows"] if r["split"] == "train"],
    "val": [r for r in fold["rows"] if r["split"] == "val"],
    "test": list(fold["test_rows"]),
}
totals = {s: len(rs) for s, rs in rows_by_split.items()}
grand = sum(totals.values())

table = []
for split in ("train", "val", "test"):
    entry = {"split": split}
    for d in DATASETS:
        n_par = len((alloc.get(split) or {}).get(d, []))
        n_til = sum(1 for r in rows_by_split[split] if r["dataset"] == d)
        entry[f"{d} parents"] = n_par
        entry[f"{d} tiles"] = n_til
    entry["tiles"] = totals[split]
    entry["share"] = round(totals[split] / grand, 4)
    entry["target"] = steel_cfg[f"{split}_frac"]
    table.append(entry)
print(pd.DataFrame(table).to_string(index=False))
print(f"\n{grand} tiles pooled from {DATASETS}")

print("\nparent allocation in full (a parent's tiles never straddle two splits):")
for split in ("train", "val", "test"):
    for d in DATASETS:
        plist = (alloc.get(split) or {}).get(d, [])
        print(f"  {split:<5} {d:<7} ({len(plist)}): {', '.join(plist) or '--'}")

## Cell 5 — fold statistics, and a `pos_weight` measured on this fold's own train split

`tiling.steel_combined_statistics` builds the `configs/fold_stats.yaml` entry.
It calls `fold_entry_statistics`, the same function that summarises every LODO
fold, so `pos_weight` and the sampler weights here are measured the one way
they are measured everywhere: `pos_weight` is `(1 − mean boundary fraction) /
mean boundary fraction` over **this fold's train rows**. Nothing is inherited
from Steel1's or Steel2's value in any other fold's entry.

The cell demonstrates that rather than asserting it in prose: it recomputes
`pos_weight` from the raw train rows, and separately computes what a
Steel1-only and a Steel2-only split would have given, so the three numbers can
be seen to differ.

`pos_weight` is also checked against `tiling.sane_pos_weight_band(line_width_px)`
— the band scaled by the width that actually produced these boundary maps
(4 px here), not the raw constant, which is calibrated at 2 px. A thicker line
covers proportionally more boundary pixels, which lowers `pos_weight` roughly
inversely with width.

In [ ]:
entry = tiling.steel_combined_statistics(fold, settings)

train_rows = [r for r in fold["rows"] if r["split"] == "train"]
recomputed = tiling.pos_weight(train_rows)
lo, hi = tiling.sane_pos_weight_band(line_width)

# What a single-dataset split of the same tiles would have given. These are
# what "inherited from Steel1 or Steel2" would look like; the fold's own value
# is measured over the mixture and is none of them.
per_dataset_pw = {
    d: tiling.pos_weight([r for r in train_rows if r["dataset"] == d])
    for d in DATASETS
}

print(f"pos_weight (recorded)      {entry['pos_weight']}")
print(f"pos_weight (recomputed)    {round(recomputed, 3)}   <- from fold['rows'], independently")
print(f"sane band at {line_width} px width  {lo:.3f} .. {hi:.3f}")
for d, pw in per_dataset_pw.items():
    print(f"  {d}-only would have been  {None if pw is None else round(pw, 3)}")
print()
print(f"train  {entry['n_train_tiles']:>5} tiles / {entry['n_train_parents']:>2} parents "
      f"from {entry['train_datasets']}")
print(f"val    {entry['n_val_tiles']:>5} tiles / {entry['n_val_parents']:>2} parents "
      f"from {entry['val_datasets']}")
print(f"test   {entry['test']['n_tiles']:>5} tiles / {entry['test']['n_parents']:>2} parents "
      f"from {entry['test']['datasets']}")
print(f"\nheld_out label: {entry['held_out']!r}   alias_of: {entry['alias_of']!r}")

print("\nboundary fraction:")
print(pd.DataFrame([
    dict(split="train", **entry["train_boundary_fraction"]),
    dict(split="val", **entry["val_boundary_fraction"]),
    dict(split="test", **entry["test"]["boundary_fraction"]),
]).to_string(index=False))

sm = entry["sampling"]
print(f"\nsampling weights (mode {sm['mode']}):")
print(pd.DataFrame([
    {"dataset": d, "parents": sm["n_parents"][d], "raw tiles": sm["raw_counts"][d],
     "weight": sm["weights"][d], "effective tiles/epoch": sm["effective_counts"][d]}
    for d in sm["raw_counts"]]).to_string(index=False))

## Cell 6 — snapshot what exists, then write this fold's two manifests

Before anything is written, the cell records a SHA-256 of every file already in
`reports/manifests/` and of `configs/fold_stats.yaml`. The checks cell later
compares against that snapshot, so "existing folds are unaffected" is a
measurement taken on this host, not a claim.

Then it writes exactly two manifests, in the same format and through the same
`tiling.write_manifest` every other fold uses:

- `fold_steel_combined.csv` — train and val rows, `split` tagged per row
- `fold_steel_combined_test.csv` — the held-out test slice, its own file

The test slice gets its own file and is never merged into `test.csv`. The two
answer different questions — domain shift onto an unseen microscope, versus
generalisation to unseen parents of the same two microscopes — and a single
file would let them be confused downstream. Nothing in training or checkpoint
selection ever reads `fold_steel_combined_test.csv`; only the evaluation cell
near the end of this notebook does.

`tiling.write_fold_report` writes `reports/tiling_fold_steel_combined.{md,json}`
— beside `reports/tiling.md`, never over it, because that file describes all
five datasets and regenerating it from a two-dataset index would silently
shrink it.

In [ ]:
import hashlib


def _sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
FOLD_STATS = CONFIGS / "fold_stats.yaml"

# The "before" picture. Everything already here belongs to another fold and
# must come out of this notebook unchanged.
before = {p.name: _sha(p) for p in sorted(MANIFEST_DIR.glob("*.csv"))}
before_fold_stats = _sha(FOLD_STATS) if FOLD_STATS.is_file() else None
print(f"{len(before)} manifest(s) already in {MANIFEST_DIR}:")
for name, digest in before.items():
    print(f"  {name:<34} {digest[:16]}")
print(f"\nconfigs/fold_stats.yaml            {(before_fold_stats or 'MISSING')[:16]}")

import yaml

before_doc = yaml.safe_load(FOLD_STATS.read_text()) if FOLD_STATS.is_file() else {}
before_folds = {k: yaml.safe_dump(v, sort_keys=True)
                for k, v in (before_doc.get("folds") or {}).items()}
print(f"folds already recorded: {sorted(before_folds)}")

train_val_path = tiling.write_manifest(fold["rows"], MANIFEST_DIR / f"{FOLD}.csv")
test_path = tiling.write_manifest(fold["test_rows"],
                                  MANIFEST_DIR / f"{FOLD}_test.csv")
manifests = {FOLD: str(train_val_path), f"{FOLD}_test": str(test_path)}

md_path, json_path = tiling.write_fold_report(fold, entry, index, manifests, REPORTS)

print()
for label, path in (("train+val", train_val_path), ("test", test_path),
                    ("report md", md_path), ("report json", json_path)):
    print(f"wrote {label:<12} {path}")

## Cell 7 — merge the fold into `configs/fold_stats.yaml`

This is the one cell that edits a file other folds also live in, so it is the
one worth understanding before running.

`tiling.write_fold_stats` — what `03_tiling.ipynb` calls — builds the entire
document from one run's stats and `write_text`s it. From this notebook's
two-dataset index that would produce a `fold_stats.yaml` containing this fold
and nothing else, deleting `fold_MetalDam`, `fold_uhcs1`, `fold_uhcs2`, `dev`
and `test`. It would not raise. It would surface much later, somewhere else, as
`fold 'dev' is not in configs/fold_stats.yaml`.

`tiling.merge_fold_stats` is the read-modify-write that makes a partial run
safe. It touches only the fold keys it is given, and refuses rather than
guesses when:

- the file does not exist (there is nothing to merge into — run the full
  `03_tiling.ipynb` once first)
- the file's recorded `patch_size`, `stride`, `min_boundary_frac` or `seed`
  disagree with this run's. Every fold in one `fold_stats.yaml` must describe
  tiles cut the same way; merging a fold cut one way into a file describing
  folds cut another would put two incompatible definitions of a tile in one
  file and nothing downstream would notice.

It returns a summary of which keys were added, which replaced and which left
alone, printed below — so the additive-ness is reported, not assumed. Re-running
this notebook replaces only this fold's key, which is why `replaced` rather than
`added` on a second run is correct and not a warning.

In [ ]:
stats_path, summary = tiling.merge_fold_stats({FOLD: entry}, settings,
                                              configs_dir=CONFIGS)

print(f"merged into {stats_path}")
print(f"  added     {summary['added']}")
print(f"  replaced  {summary['replaced']}")
print(f"  untouched {summary['untouched']}")
print(f"  {summary['n_folds_after']} fold key(s) in the file now")

after_doc = yaml.safe_load(stats_path.read_text())
after_folds = {k: yaml.safe_dump(v, sort_keys=True)
               for k, v in after_doc["folds"].items()}

changed = sorted(k for k in before_folds
                 if k != FOLD and before_folds[k] != after_folds.get(k))
print(f"\npre-existing fold entries that CHANGED: {changed or 'none'}")
if changed:
    raise SystemExit(f"merge altered fold entries it should not have: {changed}")

print(f"\nfolds.{FOLD} as training will read it:")
for key in ("held_out", "alias_of", "pos_weight", "n_train_tiles", "n_val_tiles",
            "train_datasets", "val_datasets"):
    print(f"  {key:<16} {after_doc['folds'][FOLD][key]}")

## Cell 8 — CHECKS for the fold half

Nothing in this project is verified locally, so this is where the fold's
correctness is actually established. Each check prints PASS or FAIL on its own
line and the cell raises at the end if any failed — a FAIL here means the
manifests and `fold_stats.yaml` entry just written must not be trained on.

What is checked, and why each one is worth a line of its own:

1. **No parent in two splits.** The leakage the parent-level split exists to
   prevent. Checked on the manifests as written to disk, not on the in-memory
   allocation, because the manifest is what training actually reads.
2. **Every split contains both datasets.** The `min_parents_per_split`
   guarantee, verified rather than trusted. A split missing Steel2 would make
   this fold quietly a Steel1 experiment.
3. **`pos_weight` is this fold's own.** Recomputed from the train rows and
   compared to the recorded value, and shown to differ from what either
   dataset alone would have given.
4. **Determinism.** The whole allocation is rebuilt from the same index and the
   same seed; the parent assignment must be identical.
5. **Existing folds byte-identical.** Every manifest that existed before cell 6
   is re-hashed and compared, and every pre-existing key in `fold_stats.yaml`
   is compared as normalised YAML.
6. **Manifests well-formed.** Loaded through `src.dataset.load_manifest`, the
   same validator training uses, including its refusal of any row with non-zero
   pad.

In [ ]:
from src import dataset as ds

results = []


def check(name, ok, detail=""):
    results.append((name, bool(ok), detail))
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f"  -- {detail}" if detail else ""))


print("fold_steel_combined -- checks\n")

# 1. no parent in more than one split, measured on the manifests as written
train_rows_m = ds.load_manifest(train_val_path, split="train")
val_rows_m = ds.load_manifest(train_val_path, split="val")
test_rows_m = ds.load_manifest(test_path, split="test")
parents = {
    "train": {(r["dataset"], r["parent_id"]) for r in train_rows_m},
    "val": {(r["dataset"], r["parent_id"]) for r in val_rows_m},
    "test": {(r["dataset"], r["parent_id"]) for r in test_rows_m},
}
overlaps = {f"{a}&{b}": sorted(parents[a] & parents[b])
            for a, b in (("train", "val"), ("train", "test"), ("val", "test"))}
bad = {k: v for k, v in overlaps.items() if v}
check("no parent appears in more than one split", not bad,
      "clean" if not bad else f"LEAK {bad}")

# tile ids too: a tile may not be in two manifests
ids = {s: {r["tile_id"] for r in rs} for s, rs in
       (("train", train_rows_m), ("val", val_rows_m), ("test", test_rows_m))}
tile_overlap = (ids["train"] & ids["val"]) | (ids["train"] & ids["test"]) | (ids["val"] & ids["test"])
check("no tile_id appears in more than one split", not tile_overlap,
      f"{len(tile_overlap)} shared" if tile_overlap else "clean")

# 2. every split carries parents from BOTH datasets
for split in ("train", "val", "test"):
    present = sorted({d for d, _ in parents[split]})
    counts_here = {d: sum(1 for x, _ in parents[split] if x == d) for d in DATASETS}
    check(f"{split} contains parents from both datasets",
          set(present) == set(DATASETS), str(counts_here))

# 3. pos_weight is measured on THIS fold's train split
recorded = after_doc["folds"][FOLD]["pos_weight"]
check("pos_weight matches a fresh recomputation from the train rows",
      recorded is not None and abs(recorded - round(recomputed, 3)) < 1e-6,
      f"recorded {recorded}, recomputed {round(recomputed, 3)}")
distinct = all(pw is None or abs(round(pw, 3) - recorded) > 1e-6
               for pw in per_dataset_pw.values())
check("pos_weight is not simply Steel1's or Steel2's own value", distinct,
      ", ".join(f"{d}={None if p is None else round(p, 3)}"
                for d, p in per_dataset_pw.items()) + f" vs fold {recorded}")
check("pos_weight is inside the width-scaled sane band",
      recorded is not None and lo <= recorded <= hi,
      f"{lo:.3f} <= {recorded} <= {hi:.3f}")

# 4. determinism -- same index, same seed, same assignment
again = tiling.build_steel_combined_fold(index, settings)
check("re-running the split reproduces identical parent assignments",
      again["parent_allocation"] == fold["parent_allocation"],
      "identical")
check("re-running reproduces identical tile membership",
      {r["tile_id"] for r in again["rows"]} == {r["tile_id"] for r in fold["rows"]}
      and {r["tile_id"] for r in again["test_rows"]} == ids["test"],
      "identical")

# 5. nothing that existed before this notebook changed
after_manifests = {p.name: _sha(p) for p in sorted(MANIFEST_DIR.glob("*.csv"))}
mutated = sorted(n for n, h in before.items() if after_manifests.get(n) != h)
check("pre-existing manifests are byte-identical", not mutated,
      "unchanged: " + ", ".join(before) if not mutated else f"CHANGED {mutated}")
new_files = sorted(set(after_manifests) - set(before))
check("only this fold's manifests were added",
      set(new_files) <= {f"{FOLD}.csv", f"{FOLD}_test.csv"}, f"added {new_files}")
changed_entries = sorted(k for k in before_folds
                         if k != FOLD and before_folds[k] != after_folds.get(k))
check("pre-existing fold_stats.yaml entries are unchanged", not changed_entries,
      "unchanged: " + ", ".join(sorted(before_folds)) if not changed_entries
      else f"CHANGED {changed_entries}")
check("every pre-existing fold key still present",
      set(before_folds) <= set(after_folds),
      f"missing {sorted(set(before_folds) - set(after_folds))}")

# 6. the manifests are what the loader expects
check("no manifest row carries non-zero pad",
      all(int(r["pad_right"]) == 0 and int(r["pad_bottom"]) == 0
          for r in train_rows_m + val_rows_m + test_rows_m), "all zero")
check("every row is 256 px patch",
      all(int(r["patch"]) == settings["patch_size"]
          for r in train_rows_m + val_rows_m + test_rows_m), "uniform")
achieved = {s: len(rs) / grand for s, rs in
            (("train", train_rows_m), ("val", val_rows_m), ("test", test_rows_m))}
close = all(abs(achieved[s] - steel_cfg[f"{s}_frac"]) < 0.10 for s in achieved)
check("achieved tile ratio is within 0.10 of target", close,
      " ".join(f"{s}={achieved[s]:.3f}(target {steel_cfg[f'{s}_frac']})" for s in achieved))

failed = [n for n, ok, _ in results if not ok]
print(f"\n{len(results) - len(failed)}/{len(results)} checks passed")
if failed:
    raise SystemExit("FAILED: " + "; ".join(failed))
print("fold is well-formed and additive -- safe to train on")

## Cell 9 — run the fold's unit tests on this host

`tests/test_tiling.py` is written but never run locally, so this is the first
time it executes. It covers the same invariants as the cell above plus the ones
that need synthetic inputs to provoke: a `min_parents_per_split` too large for
a dataset's parent count, fractions that do not sum to 1, and the byte-identity
of every pre-existing manifest when the pipeline is run with and without this
fold.

**A skip is not a pass.** Tests that need real data skip themselves when it is
absent, so a green run full of skips would mean nothing was actually checked.
This cell raises on any skip as well as on any failure, and names what skipped.

In [ ]:
# A skip is not a pass. The one legitimate exception is the byte-identity
# INTEGRATION test, which re-runs the whole pipeline over all five datasets and
# skips itself when their ground truth is not attached on this host -- and the
# property it checks was already measured directly, against this host's real
# files, in the checks cell above. Acknowledge that case here rather than
# letting any skip through silently.
ALLOW_SKIPS = False

proc = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_tiling.py", "-q", "-rs"],
    cwd=REPO, capture_output=True, text=True)
print(proc.stdout[-6000:])
if proc.stderr.strip():
    print("--- stderr ---")
    print(proc.stderr[-2000:])

tail = proc.stdout.strip().splitlines()[-1] if proc.stdout.strip() else ""
skipped = [ln for ln in proc.stdout.splitlines() if ln.startswith("SKIPPED")]

print(f"\nexit code {proc.returncode}")
if proc.returncode != 0:
    raise SystemExit(f"tests/test_tiling.py FAILED: {tail}")
if skipped:
    print(f"\n{len(skipped)} test(s) SKIPPED -- a skip is NOT a pass:")
    for ln in skipped:
        print(f"  {ln}")
    if not ALLOW_SKIPS:
        raise SystemExit(
            f"{len(skipped)} test(s) skipped. If every one of them is the "
            "all-datasets integration test and this host has only Steel1/Steel2 "
            "ground truth attached, set ALLOW_SKIPS = True and re-run this cell "
            "-- cell 8 already checked that property against the real files. "
            "Any OTHER skip means a test could not find something it needs; fix "
            "the environment rather than accepting the green.")
    print("  acknowledged via ALLOW_SKIPS")
print(f"\nPASS -- {tail}")

## Cell 10 — push the fold before training starts

The fold is pushed on its own, before a single epoch runs. On Kaggle
`/kaggle/working` does not survive the session, and a training run that stops at
the session budget would otherwise take the manifests and the `fold_stats.yaml`
entry down with it — and they are the definition of the experiment, not an
output of it.

`scripts/push_results.py` stages only `reports/`, `configs/` and `notebooks/`;
no image and no checkpoint can reach the repo through it.

In [ ]:
from scripts.push_results import push_results

pushed = push_results(
    f"step 3: {FOLD} -- pooled Steel1+Steel2 parent-level 70/15/15 split",
    paths=PATHS,
    expect=[train_val_path, test_path, stats_path, md_path, json_path],
)
print(f"pushed to origin/{PATHS['branch']}: {pushed}")

---

# Training on `fold_steel_combined`

Everything above defined the fold. Everything below trains on it, and it is the
same `src.train.Trainer` every other fold uses — the fold name is the only
thing that differs.

## Cell 11 — host support

`src/session.py` supplies whatever this host needs to survive its own session,
and `session.for_host(PATHS)` is the single dispatch point in the project. On
Colab it returns a no-op: Drive persists, so there is nothing to arrange. On
Kaggle it stages a checkpoint out of an attached input dataset, budgets the
session against the SLOWEST epoch seen so far, and prints what must be saved by
hand.

Nothing below tests the platform. If a host needs behaviour no method here
expresses, the method goes on the base class as a no-op and is overridden —
that is what the base class is for.

In [ ]:
from src import session as session_mod

session = session_mod.for_host(PATHS)

print(f"host support : {type(session).__name__} (platform {session.platform})")
print(f"persists     : {session.persists}   time-limited: {session.time_limited}")
print()
print(session.describe())

## Cell 12 — the trainer, and what `held_out` means for THIS fold

Every hyperparameter is read, not typed: `pos_weight` and the sampler weights
come from the `fold_stats.yaml` entry cell 7 just merged, `batch_size` from
notebook 05, `num_workers` from this host's entry in `configs/dataloader.yaml`.
The cell prints the source of each value beside it.

### The one thing that genuinely behaves differently here

This fold's `held_out` is the descriptive label `"mixed(Steel1+Steel2)"`, not a
dataset name — it has no single held-out dataset in the LODO sense. That label
matches no dataset in any per-dataset metrics breakdown, **by construction**,
and that is what makes the held-out-specific paths in `src/train.py` degrade
instead of mislabelling a real dataset:

| path | LODO fold | this fold |
| --- | --- | --- |
| `Trainer._score` (checkpoint selection) | best-threshold Dice on the held-out dataset | falls back to **pooled**, and records the key as `pooled (held-out dataset absent)` so the fallback is visible in the checkpoint |
| per-epoch table `<-HELD OUT` marker | marks the held-out dataset's row | no row is marked; both datasets are trained on |
| final-epoch comparison in `write_report` | compares held-out against the others | skipped — there is no held-out row to compare |
| `evaluate_film_inference_modes` | compares FiLM conditioning modes on the held-out dataset | **raises** a clear `TrainError`. Correct: that comparison has no meaning without a held-out dataset, and raising beats silently reporting a number about the wrong thing. Do not run the FiLM inference-mode cell for this fold. |

So the checkpoint for this fold is selected on **pooled validation Dice across
Steel1 and Steel2**. That is the right criterion here — both datasets are part
of the training mixture and neither is the thing being generalised to — but it
is a different criterion from the LODO arms, and the two are not directly
comparable on that number alone.

In [ ]:
import torch

from src import losses as losses_mod
from src import model as model_mod
from src import train as train_mod

EPOCHS = None         # None -> train.epochs from the config

train_settings = train_mod.load_config()
model_settings = model_mod.load_config()
loss_settings = losses_mod.load_config()
ds_settings = ds.load_config()
fold_stats = ds.load_fold_stats(CONFIGS)
fold_entry = fold_stats["folds"][FOLD]

trainer = train_mod.Trainer(fold=FOLD, resolved=PATHS, settings=train_settings,
                            model_settings=model_settings,
                            loss_settings=loss_settings,
                            dataset_settings=ds_settings,
                            configs_dir=CONFIGS)

print(f"fold {FOLD}  (held out: {fold_entry['held_out']})")
print(f"  train {fold_entry['n_train_tiles']} tiles / {fold_entry['n_train_parents']} "
      f"parents from {fold_entry['train_datasets']}")
print(f"  val   {fold_entry['n_val_tiles']} tiles / {fold_entry['n_val_parents']} "
      f"parents from {fold_entry['val_datasets']}")
print(f"  test  {fold_entry['test']['n_tiles']} tiles / {fold_entry['test']['n_parents']} "
      f"parents -- NOT read by training or checkpoint selection")

rows_cfg = [
    ("pos_weight", losses_mod.fold_pos_weight(FOLD, fold_stats=fold_stats),
     f"configs/fold_stats.yaml folds.{FOLD}.pos_weight (this fold's train split)"),
    ("batch_size", train_settings["batch_size"],
     "configs/default.yaml train.batch_size (measured, notebook 05)"),
    ("num_workers", trainer.resolve_num_workers(), trainer.sources["num_workers"]),
    ("epochs", EPOCHS or train_settings["epochs"], "configs/default.yaml train.epochs"),
    ("lr", train_settings["lr"], "configs/default.yaml train.lr"),
    ("encoder lr", float(train_settings["lr"]) * float(train_settings["encoder_lr_scale"]),
     "train.lr * train.encoder_lr_scale"),
    ("weight_decay", train_settings["weight_decay"], "configs/default.yaml"),
    ("warmup_epochs", train_settings["warmup_epochs"], "configs/default.yaml"),
    ("grad_clip", train_settings["grad_clip"], "configs/default.yaml"),
    ("amp", train_settings["amp"], "configs/default.yaml train.amp"),
    ("seed", train_settings["seed"], "configs/default.yaml train.seed"),
    ("encoder", model_settings["encoder"], "configs/default.yaml model.encoder"),
    ("w_bce / w_dice / w_cldice",
     f"{loss_settings['w_bce']} / {loss_settings['w_dice']} / {loss_settings['w_cldice']}",
     "configs/default.yaml loss:"),
    ("config hash", trainer.hash, "sha256 of model+loss+train+dataset settings"),
]
print()
print(f"  {'value':<26}{'resolved':<22}source")
for name, value, source in rows_cfg:
    print(f"  {name:<26}{str(value):<22}{source}")

# The selection criterion, stated rather than discovered later in the log.
print(f"\ncheckpoint selection key: {trainer.best_key()!r}")
if trainer.held_out not in (fold_entry["train_datasets"] + fold_entry["val_datasets"]):
    print("  -> matches no dataset, so _score() falls back to POOLED "
          "best-threshold Dice across Steel1 and Steel2.")
    print("     That is correct for this fold and DIFFERENT from the LODO arms; "
          "do not compare that number across the two protocols directly.")

excluded = train_mod.resolve_exclusions(FOLD, train_settings, fold_entry)
print(f"\nexclusions for this fold: {excluded or 'none'}")
if model_settings.get("film", {}).get("enabled"):
    print(f"FiLM vocabulary: {trainer.film_vocabulary}")
    print("  NOTE: the FiLM inference-mode comparison cell in 06_train.ipynb "
          "raises for this fold by design -- it needs a held-out dataset.")

## Cell 13 — build the loaders, then stage and resume

`trainer.setup()` builds the datasets and loaders from
`reports/manifests/fold_steel_combined.csv`. The test manifest is not read here
and is not read anywhere in training.

`session.stage_resume()` then puts a resumable checkpoint where the trainer
looks. On Colab it is a no-op — Drive still has the file. On Kaggle it searches
the attached input datasets for `checkpoints/<fold>/last.pt`, checks each
candidate's fold and config hash, and copies the usable one into place.
`no attached checkpoint for this fold` is CORRECT for session 1 and WRONG for
any later one.

`trainer.maybe_resume()` then decides whether that checkpoint may actually be
loaded: a checkpoint from another fold was trained under a different
`pos_weight` and a different mixture, and one under a different config hash is
not the run this notebook just described. Both are refused rather than loaded.

In [ ]:
setup_summary = trainer.setup(progress=progress)
print(f"train {len(trainer.train_ds)} tiles   val {len(trainer.val_ds)} tiles")
print(f"val composition: {trainer.val_ds.composition()}")
print(f"device {trainer.device}   AMP {trainer.amp_enabled}")
print()

staging = session.stage_resume(trainer.run_name, fold=trainer.fold)
candidates = pd.DataFrame(staging["candidates"])
if not candidates.empty:
    columns = [c for c in ("source_dataset", "fold", "epoch", "config_hash",
                           "usable", "reason", "path")
               if c in candidates.columns]
    print("every checkpoint found where this host keeps them:")
    print(candidates[columns].to_string(index=False))
for item in staging["staged"]:
    print(f"\nstaged {item['name']} from {item['source_dataset']}: "
          f"epoch {item['epoch']}, hash {item['config_hash']}, {item['size_mb']} MB")
print(f"\nstaging says resumable: {staging['resumable']}"
      + (f" -- {staging['reason']}" if staging.get("reason") else ""))

print()
status = trainer.maybe_resume()
if status["resumed"]:
    print(f"RESUMING from {status['path']}")
    print(f"  {status['reason']}")
    print(f"  {status['epochs_done']} epoch(s) done; continuing at epoch {status['start_epoch']}")
    print(f"  RNG state restored: {status['rng_restored']}")
    print(f"  best so far: {status['best']}")
else:
    print(f"STARTING CLEAN -- {status['reason']}")
    print(f"  checkpoints will be written to {trainer.checkpoint_dir}")
print(f"\nconfig hash for this run: {trainer.hash}")

## Cell 14 — train

`fit()` checkpoints after every epoch and is given `session.guard(on_epoch_end)`.
On a time-limited host that guard measures each epoch and stops on an epoch
boundary while there is still time to save — using the SLOWEST epoch so far,
not the mean, because epoch duration on a shared host is not stationary and a
mean lets one slow final epoch run past the deadline.

`SESSION BUDGET REACHED` is a clean stop, not an error. If it appears: **File →
Save Version → Quick Save with 'Save output' enabled**, then next session attach
that version's output as an input alongside the two datasets and re-run this
notebook top to bottom. A notebook cannot save its own version.

The per-epoch table prints `true_fraction` and `pred_fraction` beside the
metrics because a wrong `pos_weight` does not raise — it changes the shape of
the failure. Recall collapsing while precision looks fine means too low;
recall running ahead with the predicted fraction overshooting means too high.
No F1 number tells you that as directly.

Note that no dataset row is marked `<-HELD OUT` here: both Steel1 and Steel2
are in the training mixture, and the checkpoint is selected on the pooled row.

In [ ]:
def on_epoch_end(record, trainer):
    eta = record["eta_seconds"]
    print(f"\n{'=' * 110}")
    print(f"epoch {record['epoch']:>3}/{(EPOCHS or trainer.settings['epochs']) - 1}"
          f"   {record['seconds']:.1f}s   ETA {eta / 60:.1f} min"
          f"   lr {record['lr']:.3e}"
          f"   trainable {record['trainable_params']:,}"
          f"{'  [encoder FROZEN]' if record['encoder_frozen'] else ''}"
          f"{'   <<< BEST so far' if record['is_best'] else ''}")

    loss_table = pd.DataFrame([
        {"split": split, **{k: record[split][k]
                            for k in ("total", "bce", "dice", "cldice")}}
        for split in ("train", "val")])
    print("\n  loss terms (they differ by orders of magnitude -- read them apart):")
    print("   " + loss_table.to_string(index=False,
                                       float_format=lambda v: f"{v:9.4f}").replace("\n", "\n   "))

    metric_rows = []
    entries = list(record["metrics"]["per_dataset"].items()) + [
        ("pooled (SELECTION KEY)", record["metrics"]["pooled"])]
    for name, met_entry in entries:
        for row in ("fixed", "best"):
            metrics = met_entry[row]
            metric_rows.append({
                "dataset": name if row == "fixed" else "",
                "at": row,
                "thr": metrics["threshold"],
                **{k: metrics[k] for k in train_mod.METRIC_ORDER},
                "true_frac": metrics["true_fraction"],
                "pred_frac": metrics["pred_fraction"],
                "tiles": metrics["tiles"]})
    print("\n  validation metrics -- per dataset, then the pooled row this fold "
          "selects its checkpoint on:")
    print("   " + pd.DataFrame(metric_rows).to_string(
        index=False, float_format=lambda v: f"{v:7.4f}").replace("\n", "\n   "))

    chosen = record["best_thresholds"]
    spread = (max(chosen.values()) - min(chosen.values())) if len(chosen) > 1 else 0.0
    print("\n  chosen thresholds: "
          + ", ".join(f"{k} {v:.2f}" for k, v in sorted(chosen.items()))
          + f"   spread {spread:.2f}")
    if spread >= 0.15:
        print("    ! Steel1 and Steel2 want materially different operating "
              "points. Step 7 should threshold PER DATASET, not globally.")

    probe = record["cldice_probe"]
    print(f"\n  clDice probe on {probe['tiles']} fixed val tiles: skeleton delta "
          f"{probe['skeleton_delta']:.6f} -> "
          f"{'DEGENERATE' if probe['degenerate'] else 'ACTIVE'}")


started = time.perf_counter()
stopped_early = False
try:
    trainer.fit(epochs=EPOCHS,
                on_epoch_end=session.guard(on_epoch_end),
                progress=progress)
except session_mod.SessionStopped as stop:
    stopped_early = True
    print("\n" + "=" * 72)
    print("SESSION BUDGET REACHED -- this is a clean stop, not an error")
    print("=" * 72)
    print(stop)

history = trainer.history
print(f"\ntraining {'stopped early' if stopped_early else 'finished'}: "
      f"{len(history)} epoch(s) in trainer.history, "
      f"{(time.perf_counter() - started) / 60:.1f} min this session")
print(f"best epoch {trainer.best['epoch']} by best-threshold Dice on "
      f"{trainer.best['key']}: {trainer.best['metric']:.4f} "
      f"at threshold {trainer.best['threshold']}")

print()
session.survival_report(trainer.run_name, trainer=trainer)

## Cell 15 — evaluate on this fold's own held-out test slice

This is the first and only time `fold_steel_combined_test.csv` is read. Its
parents took no part in training and no part in choosing the checkpoint, so the
numbers here are the fold's actual answer to its question: does the model
generalise to unseen parents of the same two microscopes?

The thresholds are not swept fresh here. They are the per-dataset operating
points already chosen at the best epoch, read back out of the checkpoint —
sweeping a threshold on the test slice would be choosing an operating point on
the data being reported, which is the thing a held-out slice exists to prevent.

`load_checkpoint_model` applies the same two guards `maybe_resume` does: a
checkpoint from another fold or another config hash is refused rather than
quietly evaluated.

**Run this only when training has finished.** After a `SESSION BUDGET REACHED`
stop the best checkpoint is mid-run, and a test number read off a partial run is
not the result — it is a progress check that is easy to mistake for one later.

In [ ]:
RUN_TEST_EVAL = not stopped_early   # set True by hand to peek at a partial run

if not RUN_TEST_EVAL:
    print("training stopped early -- skipping the test slice.")
    print("Save a version, resume next session, and run this cell only once "
          "the run reports RUN COMPLETE. A test number off a partial run is "
          "not this fold's result.")
else:
    state = train_mod.load_checkpoint(trainer.best_path)
    thresholds = train_mod.best_epoch_thresholds(state)
    print(f"best checkpoint: epoch {state['best']['epoch']}, "
          f"selected on {state['best']['key']} = {state['best']['metric']:.4f}")
    print(f"thresholds chosen at that epoch: "
          + ", ".join(f"{k} {v:.2f}" for k, v in sorted(thresholds.items())))

    model, _ = train_mod.load_checkpoint_model(
        trainer.best_path, model_settings, fold=FOLD,
        expected_hash=trainer.hash, device=trainer.device,
        film_vocabulary=trainer.film_vocabulary)

    test_ds = ds.TileDataset.from_manifest(
        test_path, "test", settings=ds_settings, crops=ds.load_crops(REPORTS),
        roots={"data_root": Path(PATHS["data_root"]),
               "gt_root": Path(PATHS["gt_boundaries_root"])})
    print(f"\ntest slice: {len(test_ds)} tiles, composition {test_ds.composition()}")
    if test_ds.augment:
        raise SystemExit("the test dataset is augmented; it must not be.")

    records = train_mod.evaluate_checkpoint(
        model, test_ds, thresholds, trainer.device,
        amp_enabled=trainer.amp_enabled,
        batch_size=int(train_settings["batch_size"]),
        num_workers=trainer.resolve_num_workers())

    df = pd.DataFrame(records)
    per_ds = df.groupby("dataset").agg(
        tiles=("dice", "size"), dice_mean=("dice", "mean"),
        dice_median=("dice", "median"), dice_min=("dice", "min"),
        true_frac=("true_fraction", "mean"), pred_frac=("pred_fraction", "mean"),
        true_width=("true_width_px", "mean"), pred_width=("pred_width_px", "mean"))
    print("\nper-tile Dice on the held-out test parents, by dataset:")
    print(per_ds.to_string(float_format=lambda v: f"{v:8.4f}"))
    print(f"\npooled over {len(df)} test tiles: mean Dice {df['dice'].mean():.4f}, "
          f"median {df['dice'].median():.4f}")
    print(f"predicted boundary width {df['pred_width_px'].mean():.2f} px vs "
          f"true {df['true_width_px'].mean():.2f} px "
          f"(ground truth was generated at {line_width} px)")

    print("\nworst 5 test tiles by Dice -- look at these before believing the mean:")
    print(df.nsmallest(5, "dice")[
        ["tile_id", "dataset", "dice", "true_fraction", "pred_fraction"]
    ].to_string(index=False, float_format=lambda v: f"{v:7.4f}"))

## Cell 16 — training curves: how well did it actually learn?

Read from the checkpoint's own recorded history (`last.pt`), not from memory, so this
works in a fresh session after a resume and shows every epoch ever run, not just this
session's.

Four panels, each answering a different question:

1. **Loss (train vs val).** Both should fall. Val flattening or rising while train keeps
   falling is overfitting; the best epoch is marked. The loss terms differ by orders of
   magnitude, so total is shown here and the terms are in the per-epoch table above.
2. **Validation Dice, per dataset and pooled** (at the tuned threshold). This is the
   number checkpoint selection uses (pooled here). Steel1 and Steel2 are separate lines
   because a good pooled number can hide one dataset failing.
3. **Precision vs recall.** A wrong `pos_weight` does not raise: too low shows recall
   collapsing with precision fine, too high shows recall running ahead. Watch the gap.
4. **Predicted vs true boundary fraction.** If the model emits 1% boundary where the
   truth is 5%, no F1 number says it as directly. The dashed line is the truth.

Nothing here is a verdict on its own. It shows whether training behaved; whether the
output is *correct* is the gallery in the next cell.

In [ ]:
import matplotlib.pyplot as plt

hist_state = train_mod.load_checkpoint(trainer.last_path)
hist = hist_state.get("history") or []
if not hist:
    raise SystemExit(f"{trainer.last_path} carries no history; nothing to plot.")

epochs = [h["epoch"] for h in hist]
best_epoch = hist_state.get("best", {}).get("epoch")
names = sorted(hist[0]["metrics"]["per_dataset"])


def series(getter):
    return [getter(h) for h in hist]


fig, ax = plt.subplots(2, 2, figsize=(14, 9))

a = ax[0, 0]
a.plot(epochs, series(lambda h: h["train"]["total"]), label="train")
a.plot(epochs, series(lambda h: h["val"]["total"]), label="val")
a.set_title("loss (total)"); a.set_xlabel("epoch"); a.set_yscale("log")

a = ax[0, 1]
for n in names:
    a.plot(epochs, series(lambda h, n=n: h["metrics"]["per_dataset"][n]["best"]["dice"]),
           label=n)
a.plot(epochs, series(lambda h: h["metrics"]["pooled"]["best"]["dice"]),
       "k--", label="pooled (selection key)")
a.set_title("validation Dice at the tuned threshold"); a.set_xlabel("epoch")
a.set_ylim(0, 1)

a = ax[1, 0]
for n in names:
    a.plot(epochs, series(lambda h, n=n: h["metrics"]["per_dataset"][n]["best"]["precision"]),
           label=f"{n} precision")
    a.plot(epochs, series(lambda h, n=n: h["metrics"]["per_dataset"][n]["best"]["recall"]),
           "--", label=f"{n} recall")
a.set_title("precision vs recall"); a.set_xlabel("epoch"); a.set_ylim(0, 1)

a = ax[1, 1]
for n in names:
    line, = a.plot(epochs, series(
        lambda h, n=n: h["metrics"]["per_dataset"][n]["best"]["pred_fraction"]),
        label=f"{n} predicted")
    a.axhline(hist[-1]["metrics"]["per_dataset"][n]["best"]["true_fraction"],
              color=line.get_color(), ls=":", label=f"{n} true")
a.set_title("boundary fraction: predicted vs true"); a.set_xlabel("epoch")

for a in ax.ravel():
    if best_epoch is not None:
        a.axvline(best_epoch, color="green", alpha=0.4)
    a.grid(alpha=0.3); a.legend(fontsize=8)
fig.suptitle(f"{FOLD}: {len(hist)} epochs, best epoch {best_epoch} (green line)")
plt.tight_layout(); plt.show()

last, top = hist[-1], hist[epochs.index(best_epoch)] if best_epoch in epochs else hist[-1]
print(f"train loss {hist[0]['train']['total']:.4f} -> {last['train']['total']:.4f}   "
      f"val loss {hist[0]['val']['total']:.4f} -> {last['val']['total']:.4f}")
gap = last["val"]["total"] - last["train"]["total"]
print(f"final val-train loss gap {gap:+.4f}  (large and growing = overfitting)")
print(f"epochs after the best one: {epochs[-1] - best_epoch if best_epoch is not None else 'n/a'}")
for n in names:
    m = top["metrics"]["per_dataset"][n]["best"]
    print(f"best epoch {n:<7} Dice {m['dice']:.4f}  precision {m['precision']:.4f}  "
          f"recall {m['recall']:.4f}  pred/true fraction "
          f"{m['pred_fraction']:.4f}/{m['true_fraction']:.4f}")

## Cell 17 — prediction gallery: raw image, ground truth, prediction

For each dataset it draws the **best, median and worst** tile by Dice (ranked by the
same `evaluate_checkpoint` the rest of the pipeline uses), so you see the typical result
and the failures, not a cherry-picked good one. Choose the split with `GALLERY_SPLIT`:

- `"test"` — the fold's own held-out parents, never used for training or checkpoint
  selection. This is the honest picture. (Default.)
- `"val"` — used to pick the checkpoint, so slightly optimistic.

Columns, left to right:

1. **raw** — the image the model sees (per-image normalised, so contrast is stretched).
2. **ground truth** — the boundary map generated at `line_width_px` 4.
3. **probability** — the raw sigmoid output before any threshold.
4. **prediction** — thresholded at the operating point already chosen at the best epoch
   for that dataset (read from the checkpoint, never re-tuned on this data).
5. **error map** — white = correct boundary, **red = false boundary** (predicted, not in
   the truth), **blue = missed boundary**. Red and blue are what the Dice number hides.

It loads `best.pt`, not whatever weights the trainer holds now (the last epoch), and it
refuses a checkpoint from another fold or config hash. Steel2's ground truth is
auto-thresholded and known to be noisier than Steel1's, so a "wrong" Steel2 prediction
is sometimes the label being wrong rather than the model.

In [ ]:
GALLERY_SPLIT = "test"     # "test" (held-out, honest) or "val"

state = train_mod.load_checkpoint(trainer.best_path)
thresholds = train_mod.best_epoch_thresholds(state)
model, _ = train_mod.load_checkpoint_model(
    trainer.best_path, model_settings, fold=FOLD, expected_hash=trainer.hash,
    device=trainer.device, film_vocabulary=trainer.film_vocabulary)
model.eval()

if GALLERY_SPLIT == "test":
    gallery_ds = ds.TileDataset.from_manifest(
        test_path, "test", settings=ds_settings, crops=ds.load_crops(REPORTS),
        roots={"data_root": Path(PATHS["data_root"]),
               "gt_root": Path(PATHS["gt_boundaries_root"])})
elif GALLERY_SPLIT == "val":
    gallery_ds = trainer.val_ds
else:
    raise SystemExit(f"GALLERY_SPLIT must be 'test' or 'val', got {GALLERY_SPLIT!r}")
if gallery_ds.augment:
    raise SystemExit("the gallery dataset is augmented; it must not be.")

records = train_mod.evaluate_checkpoint(
    model, gallery_ds, thresholds, trainer.device, amp_enabled=trainer.amp_enabled,
    batch_size=int(train_settings["batch_size"]),
    num_workers=trainer.resolve_num_workers())
print(f"{GALLERY_SPLIT}: {len(records)} tiles, mean Dice "
      f"{np.mean([r['dice'] for r in records]):.4f}")

picks = []
for name in sorted({r["dataset"] for r in records}):
    ranked = train_mod.rank_tiles_by_dice(records, name)
    for label in ("best", "median", "worst"):
        picks.append((name, label, ranked[label]))

fig, axes = plt.subplots(len(picks), 5, figsize=(17, 3.4 * len(picks)))
for row, (name, label, rec) in zip(axes, picks):
    t = train_mod.tile_prediction(model, gallery_ds, rec["row_index"], trainer.device,
                                  amp_enabled=trainer.amp_enabled)
    thr = float(thresholds[name])
    truth = t["truth"] > 0.5
    pred = t["prob"] >= thr
    err = np.zeros(truth.shape + (3,), dtype=float)
    err[truth & pred] = (1, 1, 1)         # correct
    err[~truth & pred] = (0.9, 0.1, 0.1)  # false boundary
    err[truth & ~pred] = (0.2, 0.4, 1.0)  # missed boundary

    row[0].imshow(t["image"], cmap="gray")
    row[0].set_title(f"{name} {label}\n{t['tile_id'].split('/')[-2][:24]}", fontsize=8)
    row[1].imshow(truth, cmap="gray"); row[1].set_title(f"ground truth ({truth.mean():.3f})", fontsize=9)
    row[2].imshow(t["prob"], cmap="magma", vmin=0, vmax=1); row[2].set_title("probability", fontsize=9)
    row[3].imshow(pred, cmap="gray")
    row[3].set_title(f"prediction @ {thr:.2f} ({pred.mean():.3f})", fontsize=9)
    row[4].imshow(err)
    row[4].set_title(f"Dice {rec['dice']:.3f}   red=false  blue=missed", fontsize=9)
    for a in row:
        a.axis("off")
plt.suptitle(f"{FOLD} -- {GALLERY_SPLIT} split, best.pt epoch {state['best']['epoch']}", y=1.0)
plt.tight_layout(); plt.show()

## Cell 18 — CHECKS for the training half

The same discipline as cell 8, applied to the run: each line prints PASS or
FAIL and the cell raises if any failed. These are the assertions that catch the
failures which do not raise on their own — a checkpoint that quietly belongs to
another fold, a test slice that leaked into training, a `pos_weight` that
drifted from the one the fold recorded.

In [ ]:
results = []
print(f"{FOLD} training -- checks\n")

check("training produced at least one epoch", len(trainer.history) >= 1,
      f"{len(trainer.history)} epoch(s)")
check("last.pt exists", trainer.last_path.is_file(), str(trainer.last_path))
check("best.pt exists", trainer.best_path.is_file(), str(trainer.best_path))

state = train_mod.load_checkpoint(trainer.best_path)
check("checkpoint records this fold", state.get("fold") == FOLD,
      f"{state.get('fold')!r}")
check("checkpoint records this run's config hash",
      state.get("config_hash") == trainer.hash, f"{state.get('config_hash')}")
check("checkpoint selection key is the pooled fallback",
      "pooled" in str(trainer.best["key"]).lower(), f"{trainer.best['key']!r}")

# The test slice must never have been trained or validated on.
train_ids = {r["tile_id"] for r in trainer.train_ds.rows}
val_ids = {r["tile_id"] for r in trainer.val_ds.rows}
test_ids = {r["tile_id"] for r in ds.load_manifest(test_path, split="test")}
check("no test tile was trained on", not (train_ids & test_ids),
      f"{len(train_ids & test_ids)} overlap")
check("no test tile was validated on", not (val_ids & test_ids),
      f"{len(val_ids & test_ids)} overlap")
check("train and val do not overlap", not (train_ids & val_ids),
      f"{len(train_ids & val_ids)} overlap")

# pos_weight the loss actually carried vs what the fold recorded.
drift = trainer.pos_weight_drift()
recorded_pw = float(fold_stats["folds"][FOLD]["pos_weight"])
check("loss carried the fold's recorded pos_weight",
      abs(float(drift["in_use"]) - recorded_pw) < 1e-3,
      f"in use {drift['in_use']} vs fold_stats {recorded_pw}")
print(f"       (implied by the actual train rows: {drift['implied_by_this_split']:.3f}, "
      f"ratio {drift['ratio']:.3f}; drift is reported, never silently applied)")

check("validation covers both datasets",
      set(trainer.val_ds.composition()) == set(DATASETS),
      f"{sorted(trainer.val_ds.composition())}")

failed = [n for n, ok, _ in results if not ok]
print(f"\n{len(results) - len(failed)}/{len(results)} checks passed")
if failed:
    raise SystemExit("FAILED: " + "; ".join(failed))
print("run is well-formed")

## Cell 19 — write the report and push

`train_mod.write_report` writes `reports/train_<run>_<platform>.{json,md}`.
The filename is keyed by host on purpose: a Kaggle run and a Colab run of the
same fold write different files and so never collide when the `kaggle` branch
is merged into `main`.

`session.survival_report` says what must still be done by hand for this run to
survive on this host — on Kaggle, that is the Save Version step, which a
notebook cannot perform for itself.

The `kaggle` branch sync is deliberately NOT done here. Run it locally after
this notebook finishes, and check the diff by eye:

```
git checkout kaggle && git merge main && git push && git checkout main
git diff --stat main kaggle    # must be configs/kaggle.yaml + reports/train_*_kaggle.* ONLY
```

In [ ]:
for label, path in (("last", trainer.last_path), ("best", trainer.best_path)):
    if path.is_file():
        print(f"{label}: {path}  ({path.stat().st_size / 1024 ** 2:.0f} MB)")
    else:
        print(f"{label}: MISSING at {path}")

print()
survival = session.survival_report(trainer.run_name, trainer=trainer)

report_md, report_json = train_mod.write_report(trainer)
print(f"\nwrote {report_md}")
print(f"wrote {report_json}")

pushed = push_results(
    f"step 6: training run on {FOLD} (pooled Steel1+Steel2)",
    paths=PATHS,
    expect=[report_md, report_json],
)
print(f"\npushed to origin/{PATHS['branch']}: {pushed}")

done = trainer.history[-1]["epoch"] + 1
total = int(EPOCHS or train_settings["epochs"])
if done >= total:
    print(f"\nRUN COMPLETE -- {done}/{total} epochs")
    print(f"step 7 should load: {trainer.best_path}")
    print(f"  fold {FOLD}, config hash {trainer.hash}, epoch {trainer.best['epoch']}, "
          f"{trainer.best['key']} Dice {trainer.best['metric']:.4f}")
else:
    print(f"\nPARTIAL -- {done}/{total} epochs. Save Version with 'Save output', "
          "attach that output next session, and re-run this notebook top to bottom.")

print("\nNow, locally:")
print("  git checkout kaggle && git merge main && git push && git checkout main")
print("  git diff --stat main kaggle   # configs/kaggle.yaml + reports/train_*_kaggle.* ONLY")

## Cell 20 — region-level diagnostic on the TEST split: does the shape improve, or just Dice?

`06b_step6c.ipynb` already asks this question for the LODO folds: `decompose_error` splits
a pixel Dice into PLACEMENT (did the model find the true curves) versus THICKNESS (are they
too fat), and `evaluate_region_metrics` adds marker-controlled watershed against the
region partition the ground truth implies — ARI, VI, Panoptic Quality, over-segmentation —
because a model can place its centrelines correctly and still fragment or merge regions in
a way that breaks a downstream watershed seed. Both functions come straight from
`src/train.py`; nothing here reimplements them.

Two things are deliberately different from `06b_step6c.ipynb`'s cell, and both follow
directly from what this fold is:

1. **Scored on `test_ds`, not `trainer.val_ds`.** `06b`'s val split still influenced
   training indirectly through checkpoint selection. This fold's test parents took no
   part in training OR checkpoint selection (Cell 15 already established that with zero
   overlap) — so this is the one score in the whole notebook with nothing to second-guess.
2. **Reported per dataset, not "the held-out one".** `06b`'s cell singles out
   `trainer.held_out` because a LODO fold has exactly one. This fold's `held_out` is the
   descriptive label `"mixed(Steel1+Steel2)"`, which names no dataset — Steel1 and Steel2
   are BOTH training-side. Both `decompose_error` and `evaluate_region_metrics` already
   return a dict keyed by dataset with no concept of "the" held-out entry, so asking for
   `decomposition["Steel1"]` and `decomposition["Steel2"]` directly is not a workaround,
   it is what those functions do for every dataset already — `06b`'s cell just only ever
   looks up one of the keys they return.

The verdict vocabulary is exactly `06b`'s: **MISPLACED** (the true curves were not found —
thinning would not help), **OVER-DETECTION** (real curves found, but too many drawn),
**THICKNESS** (right curves, right places, too fat), **OFFSET** (right within tolerance,
a sub-tolerance registration shift), or **GOOD**. This is the number that answers whether
the pixel-Dice improvement from pooling Steel1+Steel2 reflects a genuinely better
region-level prior for the downstream watershed stage, or is Dice improving while the
region-level shape does not.

In [ ]:
# Built exactly as Cell 15 built it -- a fresh TileDataset from this fold's own
# held-out test manifest, never trainer.val_ds. Independent of whatever
# GALLERY_SPLIT was set to in Cell 17.
region_test_ds = ds.TileDataset.from_manifest(
    test_path, "test", settings=ds_settings, crops=ds.load_crops(REPORTS),
    roots={"data_root": Path(PATHS["data_root"]),
           "gt_root": Path(PATHS["gt_boundaries_root"])})
if region_test_ds.augment:
    raise SystemExit("the region-diagnostic dataset is augmented; it must not be.")

# Loaded fresh from best.pt via load_checkpoint_model -- the same fold/hash
# guards maybe_resume applies -- never whatever trainer.model currently holds
# (the LAST epoch, not necessarily the best one).
region_model, region_state = train_mod.load_checkpoint_model(
    trainer.best_path, model_settings, fold=FOLD, expected_hash=trainer.hash,
    device=trainer.device, film_vocabulary=trainer.film_vocabulary)
region_thresholds = train_mod.best_epoch_thresholds(region_state)

print(f"loaded {trainer.best_path}")
print(f"  epoch {region_state['epoch']}  held_out {region_state.get('held_out')!r}")
print(f"  selected on: {region_state['best']['key']} = "
      f"{region_state['best']['metric']:.4f} at threshold "
      f"{region_state['best']['threshold']}")
print(f"  scored on: fold_steel_combined's own TEST split "
      f"({len(region_test_ds)} tiles, {region_test_ds.composition()}) -- "
      "no part in training or checkpoint selection")
print(f"  per-dataset thresholds (from the best epoch's own sweep): {region_thresholds}")

region_cfg = train_settings["region_metrics"]
region_batch = int(train_settings["val_batch_size"] or train_settings["batch_size"])

region_results = train_mod.evaluate_region_metrics(
    region_model, region_test_ds, region_thresholds, device=trainer.device,
    marker_threshold=region_cfg["watershed_marker_threshold"],
    amp_enabled=trainer.amp_enabled, batch_size=region_batch, num_workers=0)

print("\nregion-level metrics on the TEST split, per dataset (mean over test tiles):")
print(pd.DataFrame(region_results).T.to_string(float_format=lambda v: f"{v:.4f}"))

# Computed BEFORE the report is written and passed IN to it, exactly as 06b's
# cell 12 does -- the committed region_metrics_<run>_<platform>.json must carry
# the full verdict for EVERY dataset scored, not just what stdout printed this
# session and lost when it ends.
DILATIONS = (1, 2, 3)
DISTANCES = (1, 2, 3, 5)
decomposition = train_mod.decompose_error(
    region_model, region_test_ds, region_thresholds, device=trainer.device,
    amp_enabled=trainer.amp_enabled, dilations=DILATIONS, distances=DISTANCES,
    batch_size=region_batch, num_workers=0)

# This fold has no single held-out dataset -- both Steel1 and Steel2 are
# training-side, so BOTH get their verdict printed, not just one keyed by
# trainer.held_out. decompose_error already returns a dict keyed by dataset
# with no concept of "the" held-out entry; nothing here forces one on it.
print("\nMISPLACED / OVER-DETECTION / THICKNESS verdict, TEST split, per dataset:")
for name in sorted(decomposition):
    d = decomposition[name]
    print(f"\n  {name}: {d['label']}")
    print(f"    pixel Dice {d['pixel_dice']:.4f}  skeleton Dice "
          f"{d['skeleton_dice']:.4f}  curve-length ratio "
          f"{d['curve_length_ratio']:.2f}  width ratio {d['width_ratio']:.2f}")
    print(f"    {d['verdict']}")

region_md, region_json = train_mod.write_region_metrics_report(
    trainer.run_name, trainer.platform, region_results,
    marker_threshold=region_cfg["watershed_marker_threshold"],
    config_hash=region_state["config_hash"], epoch=region_state["epoch"],
    reports_dir=Path(PATHS["reports_dir"]), decomposition=decomposition)
print(f"\nwrote {region_md}")
print(f"wrote {region_json}")

# The headline this cell exists to answer, printed once more on its own line
# so it cannot be missed scrolling past the tables above.
print("\n" + "=" * 72)
for name in sorted(decomposition):
    print(f"VERDICT  {name} (test split): {decomposition[name]['label']}"
          + ("  <- region-level prior did NOT improve with pixel Dice"
             if decomposition[name]["label"] != "GOOD" else
             "  <- region-level prior genuinely improved, not just pixel Dice"))
print("=" * 72)

pushed = push_results(
    f"step 6c: region-level diagnostic for {trainer.run_name} (test split)",
    paths=PATHS,
    expect=[region_md, region_json],
)
print(f"\npushed to origin/{PATHS['branch']}: {pushed}")